##### ***全局向量的词嵌入(GloVe)***
###### word2vec模型使用局部上下文窗口训练词嵌入，然而它并没有利用整个语料库的全局共现统计，也就是说某些词每次都是同时出现的频率。这样以来word2vec模型就需要多次迭代才能间接的学到全局信息。而GloVe模型则预先计算一个共现矩阵，用矩阵表示词与词之间共同出现的次数，然后直接优化一个损失函数，使词向量的点积能拟合共现概率的对数。
###### 具体来说对于任意的中心词$w_i$，我们考虑在整个语料库中它的上下文词，可以构成一个多重集（即元素不唯一的集合），记为$C_i$。然后统计这些上下文词，以及出现的次数。现在我们就得到了一个中心词对其他词的共现频率，据此我们可以构建一个共现矩阵。由跳元模型的公式：
$$q_{ij} = \frac{exp(u_j^Tv_i)}{\sum_{k\in V}exp(u_k^Tv_i)},$$
###### 如果加上全局共现统计，那么跳元模型的损失将等价于:
$$-\sum_{i\in V}\sum_{j\in V}x_{ij}log(q_{ij})$$
###### 其中$V$是词表，$x_{ij}$是中心词$w_i$与上下文词$w_j$的共现次数。我们用$x_i$表示上下文窗口中的所有上下文词的数量，也就是说表示在当前中心词下所有上下文中出现过的词的次数。其中$w_i$是中心词，也就是相当于$|{c_i}|$。设$p_{ij}$为在中心词给定的情况下，生成上下文词$w_j$的条件概率，也就是真实经验概率，记为$p_{ij} = \frac{x_{ij}}{x_i}$。显然$x_{ij} = x_ip_{ij}$将其带入跳元模型中可得：
$$-\sum_{i\in V}x_i\sum_{j\in V}p_{ij}log(q_{ij})$$
###### 所以它的集合意义变成了求真实分布和模型分布之间的交叉熵。
##### ***GloVe模型***
###### 有鉴于此，GloVe模型基于平方损失对跳元模型的损失进行了三个修改：首先使用$p_{ij}' = x_{ij}$和$q_{ij}' = exp(u_j^Tv_i)$而非概率分布，并取两者的对数，得到：$(\log p_{ij}' - \log q_{ij}')^2 = (u_j^Tv_i - \log x_{ij})^2.$接着为每个词$w_i$添加了两个标量模型参数：中心词偏置$b_i$和上下文偏置$c_i$，最后用权重函数$h(x_{ij})$替换每个损失项的权重，其中$h(x)$在[0, 1]的间隔内递增。最终损失函数优化为：
$$\sum_{i\in V}\sum_{j\in V}h(x_{ij})(u_j^Tv_i + b_i + c_j- \log x_{ij})^2$$
###### 在跳元模型中引入了真实概率$p_{ij}$，而模型的预测概率是$q_{ij}$，而它的问题在于预测概率的分母计算较为复杂，干脆直接不用，而我们真正关心的是分子能否近似真实概率。所以它使用两个新变量$p_{ij}' = x_{ij}$和$q_{ij}' = exp(u_j^Tv_i)$，这两个新变量分别表示原变量的分子，但其实GloVe不在要求输出是一个概率分布，它只是希望真实得分和预测得分能呈现出一种比例，而为了实现拟合效果，GloVe选择取对数操作，因为原始的$exp(u_j^Tv_i)$可能会很大，所以取两者的对数并相减，将其转化为一个线性回归问题，得到：$(\log p_{ij}' - \log q_{ij}')^2 = (u_j^Tv_i - \log x_{ij})^2.$而偏置项的引入是防止某些无意义但是流行的词汇共现计数偏高，加入偏置希望从全局层面进行一种抑制。而权重函数则用来控制某些高频但无信息，或者非常低频的噪声词，通常当$x < c$时，$h(x) = (x/c)^{\alpha}$，其中$\alpha$常取0.75, c=100,而其他情况下$h(x) = 1$，这种方式让高频词权重接近1，低频词权重小，零共现的词直接忽略。<br>而根据公式可知GloVe拟合的是一个对称概率，因为当词$w_i$出现在词$w_j$的上下文窗口时，词$w_j也出现在词$w_i$的上下文窗口。也就是说任意词的中心词向量和上下文词向量在数学上是等价的。但在实际应用中，由于初始值不同，同一个词经过训练后，在这两个向量中可能得到不同的值：GloVe将它们相加作为输出向量。